This notebook contains code for simulation of the dataset, which is analyzed and visualized in the notebook *02_visualization_and_analysis*.

# 1. Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math
from scipy import stats

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error as mse
from sklearn.metrics import r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from ipywidgets import interact, IntSlider, Dropdown, fixed

from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.cm as cm
import pickle


np.random.seed(42)

# 2. Simulation - data generation and model estimation

First, sets of X variables correlated by $\rho$ were generated in the following way: 

$X_{i} = U\sqrt{\rho}\,  + Z_i \sqrt{1-\rho}\ , $ where:

$Z_i \sim N(0, 1)$,  
$U \sim N(0, 1).$

Analyzed correlation strengths $\rho$ included:
* $\rho$ = 0.2 - almost no multicollinearity, 
* $\rho$ = 0.6 - moderate multicollinearity, 
* $\rho$ = 0.8 - strong multicollinearity,
* $\rho$ = 0.95 - almost perfect multicollinearity.

The target Y variable was constructed in the following way: 

$Y = \beta_1 X_{1} + \beta_2 X_{2} + ... + \beta_p X_{p} + \epsilon$ , where:  
$\epsilon_i \sim i.i.d.  N(0, \sigma^2), \sigma = 3$.

Different sets of structural $\beta$ parameters were analyzed.

Datasets were generated for different sample sizes: $n$ = 50 and $n$ = 500. In each case 1000 simulations were performed. 

In [2]:
results = []

nsim = 1000
sample_sizes = [50, 500]
rhos = [0.2, 0.6, 0.8, 0.95]
components_ratios = [0.5, 0.6, 0.7, 0.8, 0.9]

betas_list = [
    [1,1,1,1,1],
    [5,0.5,0.5,0.5,0.5],
    [3,-3,3,-3,3],
    [1,2,3,4,5]
]

scenarios = ["baseline", "extra_vars", "omitted_vars"]

p_true = 5
p_model_extra = 8


for scenario in scenarios:

    if scenario == "baseline":
        betas_iter = betas_list
    else:
        betas_iter = [[1,2,3,4,5]]

    for betas in betas_iter:

        betas = np.array(betas)

        for sample_size in sample_sizes:
            for rho in rhos:
                for components_ratio in components_ratios:

                    print(f"\nSTART: scenario={scenario}, n={sample_size}, rho={rho}, comp={components_ratio}, betas={betas}")

                    r2_ols_sim, r2_pca_sim, r2_pls_sim = [], [], []


                    mse_cv_ols_sim, mse_cv_pca_sim, mse_cv_pls_sim = [], [], []

                    sign_correct_ols_sim, sign_correct_pca_sim, sign_correct_pls_sim = [], [], []
                    sign_all_correct_ols_sim, sign_all_correct_pca_sim, sign_all_correct_pls_sim = [], [], []

                    params_x_ols_sim = []
                    params_x_pca_sim = []
                    params_x_pls_sim = []

                    vif_mean_sim = []
                    vif_max_sim = []
                    ci_max_sim = []

                    sd_error_y = 3
                    sd_error_z = 1

                    for sim in range(nsim):
                        if sim % 100 == 0:
                            print(f"  sim {sim}/{nsim} (scenario={scenario}, n={sample_size}, rho={rho}, comp={components_ratio})")

                        seed = hash((sample_size, rho, sim)) % 2**32
                        np.random.seed(seed)

                        Z = np.random.normal(
                            0, sd_error_z,
                            size=(sample_size, p_model_extra + 1)
                        )

                        X_full = (
                            np.sqrt(1 - rho) * Z[:, :p_model_extra]
                            + np.sqrt(rho) * Z[:, p_model_extra][:, None]
                        )
                        # standardization of X_full
                        X_full = (X_full - X_full.mean(axis=0)) / X_full.std()

                        noise = np.random.normal(0, sd_error_y, size=sample_size)

                        if scenario == "baseline":
                            X_model = X_full[:, :p_true]
                            betas_model = betas
                            y = X_model @ betas + noise

                        elif scenario == "extra_vars":
                            X_model = X_full[:, :p_model_extra]
                            betas_model = np.concatenate(
                                [betas, np.zeros(p_model_extra - p_true)]
                            )
                            y = X_full[:, :p_true] @ betas + noise

                        elif scenario == "omitted_vars":
                            X_model = X_full[:, :3]
                            betas_model = betas[:3]
                            y = X_full[:, :p_true] @ betas + noise

                        # centring
                        y = y - y.mean()

                        n, p = X_model.shape
                        min_len = len(betas_model)

                        vif_values = [
                            variance_inflation_factor(X_model, i)
                            for i in range(p)
                        ]

                        vif_mean_sim.append(np.mean(vif_values))
                        vif_max_sim.append(np.max(vif_values))


                        xtx = X_model.T @ X_model
                        eigenvalues = np.linalg.eigvalsh(xtx)
                        eigenvalues = np.clip(eigenvalues, 1e-12, None)
                        ci_max_sim.append(np.sqrt(eigenvalues.max() / eigenvalues).max())

                        # OLS
                        model_ols = sm.OLS(y, X_model).fit()
                        params_x_ols = model_ols.params
                        params_x_ols_sim.append(params_x_ols)

                        r2_ols_sim.append(model_ols.rsquared_adj)

                        mask = betas_model != 0
                        sign_ols = np.sign(params_x_ols[:min_len][mask]) == np.sign(betas_model[mask])
                        sign_correct_ols_sim.append(np.mean(sign_ols))
                        sign_all_correct_ols_sim.append(int(np.all(sign_ols)))  # czy wszystkie znaki się zgadzają

                        # PCA 
                        pca = PCA(n_components=components_ratio)
                        X_pca = pca.fit_transform(X_model)

                        model_pca = sm.OLS(y, X_pca).fit()

                        params_pca_latent = np.array(model_pca.params)
                        params_x_pca = pca.components_.T @ params_pca_latent

                        r2_pca_sim.append(model_pca.rsquared_adj)

                        mask = betas_model != 0
                        sign_pca = np.sign(params_x_pca[:min_len][mask]) == np.sign(betas_model[mask])
                        sign_correct_pca_sim.append(np.mean(sign_pca))


                        params_x_pca_sim.append(params_x_pca)

                        # PLS (The same number of variables as in PCA)
                        n_comp = X_pca.shape[1]

                        pls = PLSRegression(n_components=n_comp)
                        pls.fit(X_model, y)

                        y_pred_pls = pls.predict(X_model).ravel()
                        params_x_pls = pls.coef_.ravel()

                        r2_pls_sim.append(r2_score(y, y_pred_pls))


                        sign_pls = np.sign(params_x_pls[:min_len][mask]) == np.sign(betas_model[mask])
                        sign_correct_pls_sim.append(np.mean(sign_pls))
                        sign_all_correct_pls_sim.append(int(np.all(sign_pls)))  # czy wszystkie znaki się zgadzają


                        params_x_pls_sim.append(params_x_pls)

                        # CV
                        kf = KFold(n_splits=5, shuffle=True, random_state=42)

                        mse_cv_ols, mse_cv_pca, mse_cv_pls = [], [], []

                        for train_idx, test_idx in kf.split(X_model):

                            X_train, X_test = X_model[train_idx], X_model[test_idx]
                            y_train, y_test = y[train_idx], y[test_idx]

                            # OLS
                            m = sm.OLS(y_train, X_train).fit()
                            y_pred = m.predict(X_test)
                            mse_cv_ols.append(mse(y_test, y_pred))

                            # PCA
                            pca_cv = PCA(n_components=components_ratio)
                            X_train_pca = pca_cv.fit_transform(X_train)
                            X_test_pca = pca_cv.transform(X_test)

                            m = sm.OLS(y_train, X_train_pca).fit()
                            y_pred = m.predict(X_test_pca)
                            mse_cv_pca.append(mse(y_test, y_pred))

                            # PLS 
                            n_comp_cv = X_train_pca.shape[1]

                            pls_cv = PLSRegression(n_components=n_comp_cv)
                            pls_cv.fit(X_train, y_train)

                            y_pred = pls_cv.predict(X_test).ravel()
                            mse_cv_pls.append(mse(y_test, y_pred))

                        mse_cv_ols_sim.append(np.mean(mse_cv_ols))
                        mse_cv_pca_sim.append(np.mean(mse_cv_pca))
                        mse_cv_pls_sim.append(np.mean(mse_cv_pls))


                    # results 
                    results.append({
                        "scenario": scenario,
                        "sample_size": sample_size,
                        "rho": rho,
                        "components_ratio": components_ratio,

                        "vif_mean": np.mean(vif_mean_sim),
                        "vif_max": np.mean(vif_max_sim),
                        "ci_max": np.mean(ci_max_sim),

                        "r2_ols": np.mean(r2_ols_sim),
                        "r2_pca": np.mean(r2_pca_sim),
                        "r2_pls": np.mean(r2_pls_sim),

                        "mse_cv_ols": np.mean(mse_cv_ols_sim),
                        "mse_cv_pca": np.mean(mse_cv_pca_sim),
                        "mse_cv_pls": np.mean(mse_cv_pls_sim),

                        "sign_correct_ols": np.mean(sign_correct_ols_sim),
                        "sign_correct_pca": np.mean(sign_correct_pca_sim),
                        "sign_correct_pls": np.mean(sign_correct_pls_sim),

                        "params_x_ols": params_x_ols_sim,
                        "params_x_pca": params_x_pca_sim,
                        "params_x_pls": params_x_pls_sim,

                        "betas_true": tuple(float(x) for x in betas)
                    })


START: scenario=baseline, n=50, rho=0.2, comp=0.5, betas=[1 1 1 1 1]
  sim 0/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 100/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 200/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 300/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 400/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 500/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 600/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 700/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 800/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)
  sim 900/1000 (scenario=baseline, n=50, rho=0.2, comp=0.5)

START: scenario=baseline, n=50, rho=0.2, comp=0.6, betas=[1 1 1 1 1]
  sim 0/1000 (scenario=baseline, n=50, rho=0.2, comp=0.6)
  sim 100/1000 (scenario=baseline, n=50, rho=0.2, comp=0.6)
  sim 200/1000 (scenario=baseline, n=50, rho=0.2, comp=0.6)
  sim 300/1000 (scenario=baseline, n=50, rho=0.2, comp=0.6)
  sim 400/1000 (scenario

In [4]:
with open("simulated_data.pkl", "wb") as f:
    pickle.dump(results, f)